In [5]:
import glob
import numpy as np
import subprocess
from scipy.io import wavfile
import matplotlib.pyplot as plt
import soundfile as sf
from collections import defaultdict
import csv

In [6]:
# SETTINGS (from your log)
# =========================
SAMPLE_RATE = 384000
DTYPE = np.int16

In [7]:
 ## 1. LOAD BCL TIMELINE
# =========================
bcl_file = glob.glob("*.bcl")[0]

events = []

with open(bcl_file, newline='', encoding="utf-8") as f:
    reader = csv.reader(f)

    headers = next(reader)  # manually read header row

    for row in reader:
        if len(row) < 4:
            continue  # skip broken rows

        event = dict(zip(headers, row))
        events.append(event)

print("BCL entries:", len(events))

BCL entries: 1118


In [8]:
# 2. LOAD DWV FILES
# =========================
dwv_files = {f: np.fromfile(f, dtype=DTYPE) for f in glob.glob("*.dwv")}

print("DWV files:", len(dwv_files))

DWV files: 97


In [9]:
# 3. BUILD TIME-BASED SIGNAL
# =========================
audio = []
last_time = None

for i, e in enumerate(events):
    t = int(e["rtime"])
    state = e["state"]

    # convert time difference into samples
    if last_time is not None:
        dt = t - last_time
        silence_samples = int(dt * SAMPLE_RATE)

        # insert silence gap
        if silence_samples > 0 and silence_samples < SAMPLE_RATE * 60:
            audio.append(np.zeros(silence_samples, dtype=DTYPE))

    # if this event corresponds to audio segment
    # (heuristic: state == 1)
    if state == "1":
        # just pick next available dwv chunk
        if dwv_files:
            _, data = dwv_files.popitem()
            audio.append(data)

    last_time = t

In [10]:
# 4. FINAL OUTPUT
# =========================
if audio:
    full_audio = np.concatenate(audio)
    sf.write("reconstructed_full.wav", full_audio, SAMPLE_RATE)
    print("Saved: reconstructed_full.wav")
else:
    print("No audio reconstructed")

Saved: reconstructed_full.wav
